In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

# load the raw data
df = pd.read_csv("data/raw/support2.csv")
print("Loaded:", df.shape)

# check for dupes before we do anything to the data
raw_dupes = df.duplicated().sum()
print("Duplicate rows in raw data:", raw_dupes)

df = df.rename(columns={'d.time': 'd_time', 'num.co': 'num_co'})
TARGET = 'hospdead'

leakage_cols = ['death', 'd_time', 'slos', 'surv2m', 'surv6m', 'prg2m', 'prg6m', 'sfdm2', 'hday']
drop_cols = [c for c in leakage_cols if c in df.columns]
df_model = df.drop(columns=drop_cols)

# tracking shape at each step
shape_log = {}
shape_log['raw'] = df.shape
shape_log['after_leakage_drop'] = df_model.shape

print(f"Dropped {len(drop_cols)} leakage/administrative columns: {drop_cols}")
print(f"Remaining shape: {df_model.shape}")



In [ ]:
# quick sanity check on the categorical columns (look for typos, weird captials etc). 
cat_check_cols = df_model.select_dtypes(include='object').columns.tolist()

print("Unique values per categorical column (excluding NaN):\n")
for c in cat_check_cols:
    vals = df_model[c].dropna().unique()
    print(f"{c} ({len(vals)} unique): {sorted(vals.tolist())[:10]}{' ...' if len(vals) > 10 else ''}")

print("\nAge range: min =", df_model['age'].min(), " max =", df_model['age'].max())
# looks fine, nothing weird here

# these can't actually be 0 or negative in real life & turn them into NaN so they get handled properly
for c in ['meanbp', 'hrt', 'resp']:
    df_model.loc[df_model[c] == 0, c] = pd.NA

for c in ['totmcst', 'dnrday']:
    df_model.loc[df_model[c] < 0, c] = pd.NA



In [ ]:
# Deal with missing values 
missing_pct = df_model.isnull().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 40])

num_cols = [c for c in df_model.select_dtypes(include='number').columns if c != TARGET]
cat_cols = df_model.select_dtypes(include='str').columns.tolist()
if not cat_cols:
    cat_cols = df_model.select_dtypes(include='object').columns.tolist()

# For columns missing a ton of data, add a flag column before filling it in
# (the fact it was missing might matter, don't want to lose that info)
for c in num_cols:
    if df_model[c].isnull().any():
        if missing_pct.get(c, 0) > 40:
            df_model[f'{c}_was_missing'] = df_model[c].isnull().astype(int)
        df_model[c] = df_model[c].fillna(df_model[c].median())

# Fill with "Missing" 
for c in cat_cols:
    if df_model[c].isnull().any():
        df_model[c] = df_model[c].fillna('Missing')

# Double check data
assert df_model.isnull().sum().sum() == 0
shape_log['after_missing_handling'] = df_model.shape
print("All missing values resolved. Shape:", df_model.shape)


In [ ]:
# Check for extreme values
for c in ['age', 'charges', 'totcst', 'num_co']:
    q1, q99 = df_model[c].quantile([0.01, 0.99])
    print(f"{c}: 1st pct={q1:.1f}, 99th pct={q99:.1f}, max={df_model[c].max():.1f}")

# Check dupes again since dropping columns could've created some
post_clean_dupes = df_model.duplicated().sum()
print("\nDuplicate rows after leakage removal + missing-value handling:", post_clean_dupes)
